# AWS Bedrock AgentCore - Execute Command Demo

This notebook demonstrates how to:
1. Create and deploy a Bedrock AgentCore agent
2. Invoke the agent with standard prompts
3. Execute system commands directly in the agent runtime using [`invoke_agent_runtime_command`](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore/client/invoke_agent_runtime_command.html)

### Tutorial Details

| Information         | Details                                                   |
|:--------------------|:----------------------------------------------------------|
| Tutorial type       | Execute command on Runtime                                |
| Tool type           | HTTP server                                               |
| Tutorial components | Hosting on AgentCore Runtime                              |
| Tutorial vertical   | Cross-vertical                                            |
| Example complexity  | Medium                                                    |
| SDK used            | Amazon BedrockAgentCore Python SDK                        |


## Step 1: Install Dependencies

Install the required Python packages using uv package manager.

## Quick Start Guide

**Running this notebook:**
1. Run cells sequentially from top to bottom
2. After creating the agent files (Step 2), **restart the kernel**
3. Continue with Step 3 onwards

**What you'll learn:**
- How to deploy a Bedrock AgentCore agent from a Jupyter notebook
- How to invoke agents using different methods
- **How to execute shell commands directly in the agent runtime** ⭐

In [ ]:
!uv pip install -Uq -r requirements.txt

**⚠️ Important**

- After running the `pip install cell`, restart the kernel to ensure the libraries are correctly installed.
- `invoke_agent_runtime_command` was introduced in boto3 version `1.42.69`**`, so make sure you're running with proper version.

In [ ]:
!uv pip freeze | grep -i boto3

## Step 2: Define Agent Code

Create the agent entry point file. This agent uses:
- **BedrockAgentCoreApp**: The framework for building AgentCore applications
- **Strands Agent**: The AI agent that will process user prompts

In [ ]:
%%writefile agents/agent.py
# Import required libraries for Bedrock AgentCore
from bedrock_agentcore import BedrockAgentCoreApp
from strands import Agent

# Initialize the AgentCore application
app = BedrockAgentCoreApp()

# Create the AI agent instance
agent = Agent()

@app.entrypoint
def invoke(payload, context):
    """
    Main entry point for the agent.
    
    Args:
        payload: Dictionary containing the 'prompt' key with user input
        context: Runtime context information
    
    Returns:
        Dictionary with the agent's response message
    """
    # Extract the user prompt from the payload
    user_message = payload.get("prompt", "Hello!")
    
    # Process the message with the agent
    result = agent(user_message)
    
    # Return the response in the expected format
    return {"result": result.message}

if __name__ == "__main__":
    # Run the agent application
    app.run()

In [ ]:
%%writefile agents/requirements.txt
bedrock-agentcore
strands-agents

## Step 3: Setup AWS Configuration

Initialize AWS clients and retrieve account information needed for agent deployment.

In [ ]:
import json
import requests
import boto3
from boto3.session import Session

# Initialize boto3 session with default credentials
boto_session = Session()

# Get AWS account information
sts = boto3.client('sts')
response = sts.get_caller_identity()
account_id = response['Account']
region = boto_session.region_name

print(f"AWS Account ID: {account_id}")
print(f"AWS Region: {region}")

# Create Bedrock AgentCore client for invoking agents
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

## Step 4: Configure and Deploy Agent

Configure the agent deployment settings. The toolkit handles:
- Creating IAM execution roles automatically
- Setting up ECR repository for container images
- Deploying directly from Python code (no Docker required)

In [ ]:
import boto3
import json
import zipfile
import tempfile
import os
import time

boto_session = boto3.session.Session()
region = boto_session.region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
agentcore_control = boto3.client('bedrock-agentcore-control', region_name=region)

def create_or_get_execution_role(agent_name, region, account_id):
    """Create or retrieve an IAM execution role for the AgentCore runtime."""
    iam = boto3.client('iam')
    role_name = f"AgentCoreRuntime-{agent_name[:40]}"
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
            "Condition": {
                "StringEquals": {"aws:SourceAccount": account_id},
                "ArnLike": {"aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account_id}:runtime/*"}
            }
        }]
    }
    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {"Effect": "Allow", "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"], "Resource": "*"},
            {"Effect": "Allow", "Action": ["logs:CreateLogGroup", "logs:CreateLogDelivery", "logs:PutLogEvents",
                                            "logs:CreateLogStream", "logs:DescribeLogGroups", "logs:DescribeLogStreams"], "Resource": "*"}
        ]
    }
    try:
        role = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description=f"Execution role for AgentCore runtime {agent_name}"
        )
        iam.put_role_policy(RoleName=role_name, PolicyName="AgentCoreRuntimePermissions",
                            PolicyDocument=json.dumps(permissions_policy))
        print(f"Created IAM role: {role_name}")
        time.sleep(10)  # Allow IAM propagation
        return role['Role']['Arn']
    except iam.exceptions.EntityAlreadyExistsException:
        arn = iam.get_role(RoleName=role_name)['Role']['Arn']
        print(f"Using existing IAM role: {role_name}")
        return arn

def package_and_upload_to_s3(agent_name, files, region, account_id):
    """Package agent code as a ZIP and upload to S3 for CodeZip deployment."""
    s3 = boto3.client('s3', region_name=region)
    bucket_name = f"bedrock-agentcore-{account_id}-{region}"
    try:
        if region == 'us-east-1':
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(Bucket=bucket_name,
                             CreateBucketConfiguration={'LocationConstraint': region})
        print(f"Created S3 bucket: {bucket_name}")
    except s3.exceptions.BucketAlreadyOwnedByYou:
        print(f"Using existing S3 bucket: {bucket_name}")

    with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as tmpf:
        zip_path = tmpf.name
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in files:
            if os.path.exists(f):
                zf.write(f, os.path.basename(f))
                print(f"  Packaged: {f}")
    s3_key = f"{agent_name}/deployment.zip"
    s3.upload_file(zip_path, bucket_name, s3_key)
    os.unlink(zip_path)
    print(f"Uploaded to s3://{bucket_name}/{s3_key}")
    return bucket_name, s3_key

agent_name = "exec_cmd_sample"

# Create IAM execution role
role_arn = create_or_get_execution_role(agent_name, region, account_id)

# Package and upload agent code to S3
bucket_name, s3_key = package_and_upload_to_s3(
    agent_name,
    ["agents/agent.py", "agents/requirements.txt"],
    region,
    account_id
)
print(f"\nAgent name:  {agent_name}")
print(f"Role ARN:    {role_arn}")
print(f"S3 package:  s3://{bucket_name}/{s3_key}")


In [ ]:
# Create the AgentCore Runtime using the CodeZip deployment type
create_response = agentcore_control.create_agent_runtime(
    agentRuntimeName=agent_name,
    agentRuntimeArtifact={
        'codeConfiguration': {
            'code': {'s3': {'bucket': bucket_name, 'prefix': s3_key}},
            'runtime': 'PYTHON_3_11',
            'entryPoint': ['agents/agent.py']
        }
    },
    roleArn=role_arn,
    networkConfiguration={'networkMode': 'PUBLIC'}
)
agent_runtime_id = create_response['agentRuntimeId']
agent_runtime_arn = create_response['agentRuntimeArn']
print(f"AgentCore Runtime created:")
print(f"  Runtime ID:  {agent_runtime_id}")
print(f"  Runtime ARN: {agent_runtime_arn}")


In [ ]:
# Check the deployment status
# The agent should be in "READY" state before it can be invoked
status_response = agentcore_runtime_agent.status()
status = status_response.endpoint["status"]

print(f"Final status: {status}")

## Step 5: Test Agent Invocation

Test the deployed agent using two methods:
1. **High-level SDK method**: Using the toolkit's simplified invoke method
2. **Direct boto3 method**: Using AWS SDK for more control over the invocation

In [ ]:
# Method 1: Invoke using the high-level toolkit method
# This is the simplest way to invoke the agent
invoke_response = agentcore_runtime_agent.invoke({"prompt": "What can u do?"})
invoke_response['response']

## Step 6: Execute System Commands

The **key feature**: Execute arbitrary system commands directly in the agent runtime environment.

This uses `invoke_agent_runtime_command` which:
- Runs shell commands in the agent's containerized runtime
- Streams stdout/stderr output in real-time
- Returns exit codes and execution status
- Useful for debugging, file operations, or running scripts in the agent environment

In [ ]:
# Execute a system command in the agent runtime
# Command: List files in /tmp directory with detailed information
response = agentcore_client.invoke_agent_runtime_command(
    agentRuntimeArn=cmd_agent_arn,
    body={
        'command': '/bin/bash -c "ls -l /tmp"',  # Shell command to execute
        'timeout': 300  # Timeout in seconds (5 minutes)
    }
)

# Stream the command output
for event in response['stream']:
    if 'chunk' in event:
        chunk = event['chunk']
        print(chunk)

## Step 7: Cleanup (Optional)

Clean up AWS resources to avoid ongoing charges. This will:
- Delete the Bedrock AgentCore runtime

**⚠️ Warning**: This is destructive and cannot be undone. Only run if you're done testing.

In [ ]:
import boto3
import json

region = boto3.session.Session().region_name
agentcore_control = boto3.client('bedrock-agentcore-control', region_name=region)

# List and delete all runtimes created in this tutorial
runtimes_to_delete = agentcore_control.list_agent_runtimes().get('agentRuntimes', [])
print(f"Found {len(runtimes_to_delete)} runtimes to clean up")
for runtime in runtimes_to_delete:
    runtime_id = runtime['agentRuntimeId']
    runtime_name = runtime.get('agentRuntimeName', runtime_id)
    try:
        agentcore_control.delete_agent_runtime(agentRuntimeId=runtime_id)
        print(f"  Deleted: {runtime_name} ({runtime_id})")
    except Exception as e:
        print(f"  Failed to delete {runtime_name}: {e}")
print("Cleanup complete.")


---

## Summary

This notebook demonstrated:

1. ✅ **Agent Creation**: Deployed a Bedrock AgentCore agent with Python code
2. ✅ **Agent Invocation**: Called the agent using both high-level and low-level methods
3. ✅ **Command Execution**: Used `invoke_agent_runtime_command` to execute shell commands in the runtime environment

### Key Takeaways

- **Event Streaming**: The `invoke_agent_runtime_command` returns an event stream with three event types:
  - `contentStart`: Indicates command execution has begun
  - `contentDelta`: Contains streaming stdout/stderr output
  - `contentStop`: Provides final exit code and status
  
- **Use Cases**: This feature is useful for:
  - Running diagnostic commands
  - Executing data processing scripts
  - File system operations in the agent environment
  - Integration testing and debugging